<a href="https://colab.research.google.com/github/FatemaTabassum/Data_engineering/blob/main/classification/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Set up your environment

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# `Step 2: Import libraries and load the dataset`



In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
# Load the dataset example
data = pd.read_csv('customer_churn.csv')

In [5]:
# Explore the Dataset
print(data.head())
print(data.info())

   CustomerID  Tenure  MonthlyCharges  TotalCharges        Contract  \
0        1001       5            70.0         350.0  Month-to-month   
1        1002      10            85.5         850.5        Two year   
2        1003       3            55.3         165.9        One year   
3        1004       8            90.0         720.0  Month-to-month   
4        1005       2            65.2         130.4        One year   

      PaymentMethod  Churn  
0  Electronic check      1  
1      Mailed check      0  
2  Electronic check      1  
3       Credit card      0  
4  Electronic check      1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      5 non-null      int64  
 1   Tenure          5 non-null      int64  
 2   MonthlyCharges  5 non-null      float64
 3   TotalCharges    5 non-null      float64
 4   Contract        5 non-nu

# Step 3: Preprocess the data

Handle missing values and simplify the dataset

Identify and handle any missing values in the dataset. This could involve filling missing values, dropping rows or columns, or using imputation techniques. Removing unnecessary columns, such as CustomerID, helps keep the dataset clean. In this smaller dataset, removing these columns is essential to accurately train a model using only the important values.

In [6]:
data = data.drop(columns=['CustomerID']) #Simplify the dataset
data = data.dropna()  # Simple example of dropping missing values

In [7]:
print(data.columns)

Index(['Tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'PaymentMethod',
       'Churn'],
      dtype='object')


Encode categorical variables

Convert categorical variables into numerical format using techniques like one-hot encoding.

In [8]:
# Convert categorical variables into numerical format using techniques like one-hot encoding.
data = pd.get_dummies(data, drop_first=True)

In [15]:
print(data.head(5))

   Tenure  MonthlyCharges  TotalCharges  Churn  Contract_One year  \
0       5            70.0         350.0      1              False   
1      10            85.5         850.5      0              False   
2       3            55.3         165.9      1               True   
3       8            90.0         720.0      0              False   
4       2            65.2         130.4      1               True   

   Contract_Two year  PaymentMethod_Electronic check  \
0              False                            True   
1               True                           False   
2              False                            True   
3              False                           False   
4              False                            True   

   PaymentMethod_Mailed check  
0                       False  
1                        True  
2                       False  
3                       False  
4                       False  


In [9]:
# Split the data into training and testing sets to evaluate the model’s performance.

X = data.drop('Churn', axis=1)
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Develop and train the model

Choose and build the model

Depending on your chosen framework, build a model suited to the task of classification.

In [11]:
# tensorflow
# model = tf.keras.Sequential([
#     tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
#     tf.keras.layers.Dropout(0.5),
#     tf.keras.layers.Dense(32, activation='relu'),
#     tf.keras.layers.Dense(1, activation='sigmoid')
# ])


# Pytorch example
# Choose and build the model
class ChurnModel(nn.Module):
    def __init__(self):
        super(ChurnModel, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = nn.functional.dropout(x, 0.5, training=self.training)
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

model = ChurnModel()





# Scikit-learn example
# model = RandomForestClassifier(n_estimators=100, random_state=42)

Train the model using the training data. Monitor the training process to ensure the model is learning appropriately.

In [14]:
# Tensorflow example
# model.compile(optimizer='adam',
#               loss='binary_crossentropy',
#               metrics=['accuracy'])

# model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))







#PyTorch example
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop (simplified example)
for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    # Fix: Ensure X_train.values is converted to float32 before creating the tensor
    outputs = model(torch.tensor(X_train.values.astype(np.float32)))
    loss = criterion(outputs.squeeze(), torch.tensor(y_train.values).float())
    loss.backward()
    optimizer.step()






# Scikit-learn example
# model.fit(X_train, y_train)

# Step 5: Evaluate and optimize the model



Evaluate the model’s performance

After training, evaluate the model using the test data. Consider metrics such as accuracy, precision, recall, and F1 score to assess the model’s performance.

In [15]:
# tensorflow example
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)
print(f'Test accuracy: {test_acc}')

AttributeError: 'ChurnModel' object has no attribute 'evaluate'

In [18]:
model.eval()
outputs = model(torch.tensor(X_test.values.astype(np.float32)))
predictions = (outputs.squeeze().detach().numpy() > 0.5).astype(int)
accuracy = np.mean(predictions == y_test.values)
print(f'Test accuracy: {accuracy}')

Test accuracy: 0.0


In [19]:
# Scikit-learn example
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f'Test accuracy: {accuracy}')

AttributeError: 'ChurnModel' object has no attribute 'predict'

Optimize for deployment

Optimize the model for deployment by considering techniques such as model pruning, quantization, or simplifying the model architecture without sacrificing too much accuracy. This step is crucial for ensuring the model can be deployed efficiently in a business environment.

In [ ]:
# TensorFlow example
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

In [20]:
# PyTorch example
# Apply dynamic quantization
quantized_model = torch.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

/tmp/ipykernel_18938/952410338.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [ ]:
# Scikit-learn example
# Simplify model by limiting its maximum depth
pruned_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10, max_features='sqrt')

pruned_model.fit(X_train, y_train)
pruned_predictions = pruned_model.predict(X_test)
pruned_accuracy = accuracy_score(y_test, pruned_predictions)
print(f'Pruned Test accuracy: {pruned_accuracy}')

# **Step** 6: Prepare the model for deployment

Save the model

Save the trained model so it can be deployed in the production environment.



In [ ]:
# TensorFlow example
model.save('churn_model.h5')

In [21]:
# PyTorch example
torch.save(model.state_dict(), 'churn_model.pth')

In [ ]:

import joblib
joblib.dump(model, 'churn_model.pkl')

Document the model and deployment instructions

Provide documentation that includes details about the model, how to load it, and any necessary steps for deploying it into the business environment. This documentation is crucial for ensuring smooth integration and maintenance.

